In [ ]:
#import Library 
import matplotlib.pyplot as plt
import numpy as np
import SimpleITK as sitk
from scipy import ndimage as ndi
from pathlib import Path
import os
import glob

from utils.helper import image_save, convert_nrrd_to_niigz

In [ ]:
# Path Setup
MAIN_PATH       = Path('./').resolve()
DATA_PATH = MAIN_PATH / 'dataset' 
RESULT_PATH = MAIN_PATH / 'result' 

In [ ]:
def resize_image_3d(input_filepath, output_filepath):
    # Load the image
    img = sitk.ReadImage(input_filepath,sitk.sitkInt32)
    
    # Get the original image size
    original_size = img.GetSize()
    print("Original image size:", original_size)

    new_size=(256, 512, original_size[2])

    print("New size:", new_size)
    # Compute the resampling factors for each axis
    resampling_factors = [new_size[i] / original_size[i] for i in range(3)]

    print("Rresampling factors:", resampling_factors)

    # Define the resampling method (interpolation)
    interpolator = sitk.sitkBSpline

    # Perform the resampling
    resized_img = sitk.Resample(img, new_size, sitk.Transform(), interpolator, img.GetOrigin(), resampling_factors, img.GetDirection(), 0, img.GetPixelID())
    print("Resampled image size:", resized_img.GetSize())

    # Write the resized image to file
    sitk.WriteImage(resized_img, output_filepath)

    return resized_img

def display_image_and_histogram(image):

    # Select the middle slice along the first axis
    slice_idx = image.shape[0] // 2
    slice_2d = image[slice_idx, :, :]
    
    # Display the 2D slice
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.title("2D Slice")
    plt.imshow(slice_2d, cmap='gray')
    plt.axis('off')

    # Display the histogram of the 2D slice
    plt.subplot(1, 2, 2)
    plt.title("Histogram of 2D Slice")
    plt.hist(slice_2d.flatten(), bins=50, color='c', edgecolor='black')
    plt.xlabel('Pixel intensity')
    plt.ylabel('Frequency')

    plt.tight_layout()
    plt.show()


In [ ]:
# Function to load dataset
def load_dataset(data_path):
    dataset = []
    projects = [folder for folder in data_path.iterdir() if folder.is_dir()]
    for project in projects:
        if project.name == 'Wdr47Kusss':  # Check if the project is 'Wdr47Kusss'
            print(project)
            rc_files = glob.glob(os.path.join(project, "*_RC.nrrd"))
            brainmask_files = glob.glob(os.path.join(project, "*_brainmask.seg.nrrd"))
            for rc_file, brainmask_file in zip(rc_files, brainmask_files):
                print(rc_file)
                print(brainmask_file)
                output_filepath = Path(rc_file).with_suffix('.nii.gz')
                rc_image = convert_nrrd_to_niigz(Path(rc_file), output_filepath)  # Convert NRRD to NIfTI

                output_filepath = Path(brainmask_file).with_suffix('.nii.gz')
                brainmask_image = convert_nrrd_to_niigz(Path(brainmask_file), output_filepath)  # Convert NRRD to NIfTI

    return dataset
dataset = load_dataset(DATA_PATH)

In [ ]:
# image convert .nrrd to nii.gz usage
input_filepath = DATA_PATH / 'NG4120_brainmask.seg.nrrd'
output_filepath = DATA_PATH / 'NG4120_brainmask.seg.nii.gz'

seg_image = convert_nrrd_to_niigz(input_filepath, output_filepath)

In [ ]:
# image resize

input_filepath = DATA_PATH / 'NG4120_brainmask.seg.nii.gz'
output_filepath = DATA_PATH /  '512x256' / 'NG4120_brainmask_512x256.nii.gz'

seg_image = resize_image_3d(input_filepath, output_filepath)

resampled_array = sitk.GetArrayFromImage(seg_image)

display_image_and_histogram(resampled_array)

In [ ]:
# image convert .nrrd to nii.gz usage
input_filepath = DATA_PATH / 'NG4120_RC.nrrd'
output_filepath = DATA_PATH / 'NG4120_RC.nii.gz'

seg_image = convert_nrrd_to_niigz(input_filepath, output_filepath)

In [ ]:
input_filepath = DATA_PATH / 'NG4120_RC.nii.gz'
output_filepath = DATA_PATH /  '512x256' / 'NG4120_RC_512x256.nii.gz'

org_image = resize_image_3d(input_filepath, output_filepath)

resampled_array = sitk.GetArrayFromImage(org_image)

display_image_and_histogram(resampled_array)